# Fractured Knee Dataset Exploration (Ruikar et al.)

Exploratory data analysis of the fractured knee CT dataset before any preprocessing.

**Objectives:**
1. Catalogue all available cases and extract DICOM metadata
2. Filter out invalid cases (scouts/localizers with fewer than 10 Z-slices)
3. Visualize anatomy coverage (axial, coronal, sagittal views) for all valid cases
4. Analyze HU intensity distributions with the confirmed bone window [-450, 1050]
5. Assess volume dimensions and spacing variability
6. Cross-dataset comparison with VSD healthy dataset

**Input**: Raw DICOM files from `data/raw/fractured/{PartLeft,PartRight}/`  
**Output**: Exploration figures saved to `reports/figures/fractured_exploration/`

**Confirmed parameters** (shared with VSD exploration):
- Spatial resampling: 0.5mm isotropic
- HU bone window: [-450, 1050]
- Orientation: RAS (Right-Anterior-Superior)
- Target volume: config-driven (128^3 local / 512^3 HPC)


## Cell 1 - Imports & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter1d

# ============================================================
# Configuration
# ============================================================
PROJECT_ROOT = Path(r"c:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject")

FRACTURED_ROOT = PROJECT_ROOT / "data" / "raw" / "fractured"
VSD_ROOT = PROJECT_ROOT / "data" / "raw" / "VSD_Dataset"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "fractured_exploration"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Confirmed parameters
BONE_HU_THRESHOLD = 200
HU_WINDOW_MIN = -450
HU_WINDOW_MAX = 1050
TARGET_SPACING = 0.5      # mm isotropic
MIN_Z_SLICES = 10         # cases with fewer slices are scouts/localizers

print(f"Fractured root: {FRACTURED_ROOT}")
print(f"Figures dir   : {FIG_DIR}")
print(f"Bone window   : [{HU_WINDOW_MIN}, {HU_WINDOW_MAX}] HU")
print(f"Min Z-slices  : {MIN_Z_SLICES}")


## Cell 2 - Data Discovery & Metadata

Scan PartLeft and PartRight directories. Extract metadata from each DICOM series.
Flag cases with fewer than 10 Z-slices as invalid (scout/localizer images).

In [ ]:
def discover_fractured_cases(fractured_root):
    """Discover all fractured cases and extract metadata."""
    cases = []
    for part in ["PartLeft", "PartRight"]:
        part_dir = fractured_root / part
        if not part_dir.exists():
            print(f"  [WARN] Not found: {part_dir}")
            continue

        for case_dir in sorted(part_dir.iterdir()):
            if not case_dir.is_dir():
                continue

            reader = sitk.ImageSeriesReader()
            dicom_names = reader.GetGDCMSeriesFileNames(str(case_dir))
            if not dicom_names:
                cases.append({
                    "case_id": case_dir.name, "part": part,
                    "case_path": str(case_dir), "valid": False,
                    "reason": "No DICOM files",
                })
                continue

            reader.SetFileNames(dicom_names)
            try:
                img = reader.Execute()
                arr = sitk.GetArrayFromImage(img).astype(np.float32)
                sp = img.GetSpacing()
                sz = img.GetSize()

                valid = sz[2] >= MIN_Z_SLICES
                reason = None if valid else f"Only {sz[2]} Z-slice(s) (scout/localizer)"

                cases.append({
                    "case_id": case_dir.name, "part": part,
                    "case_path": str(case_dir), "valid": valid,
                    "reason": reason,
                    "size_xyz": sz,
                    "spacing_x": round(sp[0], 6),
                    "spacing_y": round(sp[1], 6),
                    "spacing_z": round(sp[2], 6),
                    "n_slices": sz[2],
                    "phys_z_mm": round(sz[2] * sp[2], 1),
                    "phys_x_mm": round(sz[0] * sp[0], 1),
                    "phys_y_mm": round(sz[1] * sp[1], 1),
                    "hu_min": float(arr.min()),
                    "hu_max": float(arr.max()),
                    "hu_mean": float(arr.mean()),
                    "img": img,
                    "arr": arr,
                })
            except Exception as e:
                cases.append({
                    "case_id": case_dir.name, "part": part,
                    "case_path": str(case_dir), "valid": False,
                    "reason": f"Read error: {e}",
                })
    return cases


print("Scanning fractured dataset...")
print()
all_frac_cases = discover_fractured_cases(FRACTURED_ROOT)

valid_cases = [c for c in all_frac_cases if c.get("valid", False)]
invalid_cases = [c for c in all_frac_cases if not c.get("valid", False)]

print(f"Total cases found : {len(all_frac_cases)}")
print(f"Valid cases       : {len(valid_cases)}")
print(f"Filtered out      : {len(invalid_cases)}")
print()

if invalid_cases:
    print("EXCLUDED CASES:")
    for c in invalid_cases:
        print(f"  {c['case_id']} ({c['part']}): {c['reason']}")
    print()

# Summary table for valid cases
display_cols = ["case_id", "part", "size_xyz", "spacing_x", "spacing_y", "spacing_z",
                "n_slices", "phys_z_mm", "hu_min", "hu_max"]
df_valid = pd.DataFrame([{k: c.get(k) for k in display_cols} for c in valid_cases])
display(df_valid)


## Cell 3 - Spacing & Size Summary

Understand the variability in pixel spacing and slice thickness across the dataset.
This informs the resampling strategy (target: 0.5mm isotropic).

In [ ]:
print("XY Spacing range:")
xy_spacings = [c["spacing_x"] for c in valid_cases]
print(f"  Min: {min(xy_spacings):.4f}mm, Max: {max(xy_spacings):.4f}mm")
print()

print("Z Spacing (slice thickness):")
z_spacings = [c["spacing_z"] for c in valid_cases]
print(f"  Min: {min(z_spacings):.4f}mm, Max: {max(z_spacings):.4f}mm")
z_thin = [c for c in valid_cases if c["spacing_z"] < 1.0]
z_thick = [c for c in valid_cases if c["spacing_z"] >= 1.0]
print(f"  Thin-slice (<1mm): {len(z_thin)} cases")
print(f"  Thick-slice (>=1mm): {len(z_thick)} cases")
print()

print("Physical Z-extent:")
z_extents = [c["phys_z_mm"] for c in valid_cases]
print(f"  Min: {min(z_extents):.0f}mm, Max: {max(z_extents):.0f}mm, Mean: {np.mean(z_extents):.0f}mm")
print()

print("Number of slices:")
n_slices = [c["n_slices"] for c in valid_cases]
print(f"  Min: {min(n_slices)}, Max: {max(n_slices)}, Mean: {np.mean(n_slices):.0f}")

# Bar chart of z-spacing per case
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

case_labels = [f"{c['case_id']}\n{c['part'][-1]}" for c in valid_cases]
colors = ["steelblue" if c["spacing_z"] < 1.0 else "coral" for c in valid_cases]

axes[0].bar(range(len(valid_cases)), [c["spacing_z"] for c in valid_cases], color=colors)
axes[0].set_xticks(range(len(valid_cases)))
axes[0].set_xticklabels(case_labels, rotation=45, ha="right", fontsize=7)
axes[0].set_ylabel("Z Spacing (mm)")
axes[0].set_title("Z Spacing Per Case (blue=thin, coral=thick)")
axes[0].axhline(y=TARGET_SPACING, color="green", linestyle="--", label=f"Target: {TARGET_SPACING}mm")
axes[0].legend()

axes[1].bar(range(len(valid_cases)), [c["phys_z_mm"] for c in valid_cases], color="steelblue")
axes[1].set_xticks(range(len(valid_cases)))
axes[1].set_xticklabels(case_labels, rotation=45, ha="right", fontsize=7)
axes[1].set_ylabel("Physical Z-extent (mm)")
axes[1].set_title("Physical Z-Extent Per Case")

plt.suptitle("Fractured Dataset - Spacing & Size Variability", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "fractured_spacing_summary.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


## Cell 4 - Per-Case Overview (4-Panel Visualization)

For each valid case, show:
1. **Axial** slice at volume midpoint
2. **Coronal** slice at volume midpoint
3. **Sagittal** slice at volume midpoint
4. **Info panel** with key metadata

This gives a quick visual assessment of each case's anatomy and fracture visibility.

In [ ]:
for case in valid_cases:
    sid = case["case_id"]
    part = case["part"]
    arr = case["arr"]
    mid_z = arr.shape[0] // 2
    mid_y = arr.shape[1] // 2
    mid_x = arr.shape[2] // 2

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Axial
    axes[0].imshow(arr[mid_z], cmap="gray", vmin=-500, vmax=1500)
    axes[0].set_title(f"Axial (z={mid_z})")
    axes[0].axis("off")

    # Coronal
    axes[1].imshow(arr[:, mid_y, :], cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[1].set_title(f"Coronal (y={mid_y})")
    axes[1].axis("off")

    # Sagittal
    axes[2].imshow(arr[:, :, mid_x], cmap="gray", vmin=-500, vmax=1500, aspect="auto")
    axes[2].set_title(f"Sagittal (x={mid_x})")
    axes[2].axis("off")

    # Info panel
    axes[3].axis("off")
    info_text = (
        f"Case: {sid} ({part})\n"
        f"Size: {case['size_xyz']}\n"
        f"Spacing: ({case['spacing_x']:.4f}, {case['spacing_y']:.4f}, {case['spacing_z']:.4f})\n"
        f"Z-extent: {case['phys_z_mm']:.0f}mm\n"
        f"N slices: {case['n_slices']}\n"
        f"HU range: [{case['hu_min']:.0f}, {case['hu_max']:.0f}]\n"
        f"HU mean: {case['hu_mean']:.0f}"
    )
    axes[3].text(0.1, 0.5, info_text, transform=axes[3].transAxes,
                fontsize=12, verticalalignment="center", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
    axes[3].set_title("Metadata")

    plt.suptitle(f"Fractured {sid} ({part}) - Overview", fontsize=14)
    plt.tight_layout()
    fig_path = FIG_DIR / f"frac_{sid}_{part}_exploration.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path}")


## Cell 5 - HU Intensity Distributions

Plot HU histograms for each valid case with the confirmed bone window [-450, 1050] marked.
This validates that the window captures the relevant bone structures in fractured cases.

In [ ]:
n_valid = len(valid_cases)
ncols = 4
nrows = (n_valid + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.ravel()

for i, case in enumerate(valid_cases):
    sid = case["case_id"]
    arr = case["arr"]

    tissue = arr[arr > -900].ravel()

    axes[i].hist(tissue, bins=200, color="steelblue", alpha=0.7, density=True)
    axes[i].axvline(x=HU_WINDOW_MIN, color="red", linestyle="--", alpha=0.7, label="Window low (-450)")
    axes[i].axvline(x=HU_WINDOW_MAX, color="red", linestyle="--", alpha=0.7, label="Window high (1050)")
    axes[i].axvline(x=BONE_HU_THRESHOLD, color="orange", linestyle=":", alpha=0.7, label="Bone thresh (200)")
    axes[i].set_title(f"{sid} ({case['part'][-1]})", fontsize=9)
    axes[i].set_xlabel("HU")
    axes[i].set_xlim(-500, 2000)
    if i == 0:
        axes[i].legend(fontsize=7)

# Hide unused subplots
for j in range(n_valid, len(axes)):
    axes[j].axis("off")

plt.suptitle("HU Intensity Distributions - Fractured Cases (air excluded, HU > -900)", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "fractured_hu_histograms.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

# Save individual histograms
for case in valid_cases:
    sid = case["case_id"]
    part = case["part"]
    arr = case["arr"]
    tissue = arr[arr > -900].ravel()

    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.hist(tissue, bins=300, color="steelblue", alpha=0.7, density=True)
    ax2.axvline(x=HU_WINDOW_MIN, color="red", linestyle="--", label="Window low (-450)")
    ax2.axvline(x=HU_WINDOW_MAX, color="red", linestyle="--", label="Window high (1050)")
    ax2.axvline(x=BONE_HU_THRESHOLD, color="orange", linestyle=":", label="Bone thresh (200)")
    ax2.set_title(f"{sid} ({part}) - HU Distribution")
    ax2.set_xlabel("HU")
    ax2.set_ylabel("Density")
    ax2.set_xlim(-600, 2500)
    ax2.legend()
    plt.tight_layout()
    fig2.savefig(FIG_DIR / f"frac_{sid}_{part}_hu_histogram.png", dpi=150, bbox_inches="tight")
    plt.close(fig2)

print(f"Individual histograms saved for {n_valid} cases.")


## Cell 6 - Axial Knee Slices (9-Slice Grid)

Visualize 9 evenly-spaced axial slices through each case to assess
fracture visibility and anatomy coverage at different z-levels.

In [ ]:
for case in valid_cases:
    sid = case["case_id"]
    part = case["part"]
    arr = case["arr"]
    n_z = arr.shape[0]

    # 9 evenly spaced slices
    slice_indices = np.linspace(0, n_z - 1, 9, dtype=int)

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes_flat = axes.ravel()

    for j, sl_idx in enumerate(slice_indices):
        axes_flat[j].imshow(arr[sl_idx], cmap="gray", vmin=-500, vmax=1500)
        pct = sl_idx / max(n_z - 1, 1) * 100
        axes_flat[j].set_title(f"slice {sl_idx} ({pct:.0f}%%)", fontsize=9)
        axes_flat[j].axis("off")

    plt.suptitle(f"{sid} ({part}) - 9 Axial Slices", fontsize=14)
    plt.tight_layout()
    fig_path = FIG_DIR / f"frac_{sid}_{part}_axial_grid.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path}")


## Cell 7 - Cross-Dataset Comparison (Fractured vs VSD Healthy)

Compare key characteristics between fractured and healthy datasets
to understand the standardization work needed.

In [ ]:
# Load VSD metadata (lightweight - only spacing/size, not full arrays)
vsd_cases_meta = []
EXCLUDE_VSD = {"010"}

for subject_name in sorted(os.listdir(VSD_ROOT)):
    subject_dir = VSD_ROOT / subject_name
    if not subject_dir.is_dir() or subject_name in EXCLUDE_VSD:
        continue

    subdirs = [d for d in subject_dir.iterdir() if d.is_dir()]
    if not subdirs:
        continue

    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(str(subdirs[0]))
    if not dicom_names:
        continue
    reader.SetFileNames(dicom_names)
    img = reader.Execute()
    sp = img.GetSpacing()
    sz = img.GetSize()

    vsd_cases_meta.append({
        "subject_id": subject_name,
        "spacing_x": round(sp[0], 6),
        "spacing_z": round(sp[2], 6),
        "n_slices": sz[2],
        "phys_z_mm": round(sz[2] * sp[2], 1),
    })

print(f"VSD valid subjects loaded: {len(vsd_cases_meta)}")
print(f"Fractured valid cases: {len(valid_cases)}")


In [ ]:
vsd_sp_xy = [c["spacing_x"] for c in vsd_cases_meta]
vsd_sp_z = [c["spacing_z"] for c in vsd_cases_meta]
vsd_z_mm = [c["phys_z_mm"] for c in vsd_cases_meta]
vsd_knees = len(vsd_cases_meta) * 2  # bilateral

frac_sp_xy = [c["spacing_x"] for c in valid_cases]
frac_sp_z = [c["spacing_z"] for c in valid_cases]
frac_z_mm = [c["phys_z_mm"] for c in valid_cases]
frac_knees = len(valid_cases)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. XY spacing
axes[0, 0].hist(vsd_sp_xy, bins=10, alpha=0.7, color="steelblue", label="VSD (healthy)")
axes[0, 0].hist(frac_sp_xy, bins=10, alpha=0.7, color="coral", label="Ruikar (fractured)")
axes[0, 0].set_title("XY Pixel Spacing (mm)")
axes[0, 0].set_xlabel("Spacing (mm)")
axes[0, 0].legend()

# 2. Z spacing
axes[0, 1].hist(vsd_sp_z, bins=10, alpha=0.7, color="steelblue", label="VSD")
axes[0, 1].hist(frac_sp_z, bins=10, alpha=0.7, color="coral", label="Ruikar")
axes[0, 1].set_title("Z Spacing / Slice Thickness (mm)")
axes[0, 1].set_xlabel("Spacing (mm)")
axes[0, 1].legend()

# 3. Number of slices
axes[0, 2].bar(["VSD"], [np.mean([c["n_slices"] for c in vsd_cases_meta])],
               yerr=[np.std([c["n_slices"] for c in vsd_cases_meta])],
               color="steelblue", alpha=0.7, capsize=5)
axes[0, 2].bar(["Ruikar"], [np.mean([c["n_slices"] for c in valid_cases])],
               yerr=[np.std([c["n_slices"] for c in valid_cases])],
               color="coral", alpha=0.7, capsize=5)
axes[0, 2].set_title("Number of Slices (mean +/- std)")
axes[0, 2].set_ylabel("Slices")

# 4. Z-extent
axes[1, 0].bar(["VSD"], [np.mean(vsd_z_mm)], yerr=[np.std(vsd_z_mm)],
               color="steelblue", alpha=0.7, capsize=5)
axes[1, 0].bar(["Ruikar"], [np.mean(frac_z_mm)], yerr=[np.std(frac_z_mm)],
               color="coral", alpha=0.7, capsize=5)
axes[1, 0].set_title("Physical Z-Extent (mm, mean +/- std)")
axes[1, 0].set_ylabel("mm")

# 5. Sample counts
axes[1, 1].bar(["VSD\n(healthy)", "Ruikar\n(fractured)"], [vsd_knees, frac_knees],
               color=["steelblue", "coral"], alpha=0.7)
axes[1, 1].set_title("Total Knee Volumes")
axes[1, 1].set_ylabel("Count")
for j, v in enumerate([vsd_knees, frac_knees]):
    axes[1, 1].text(j, v + 0.3, str(v), ha="center", fontweight="bold")

# 6. Summary
axes[1, 2].axis("off")
summary = (
    "CROSS-DATASET SUMMARY\n"
    "=====================\n"
    f"VSD (healthy):     {len(vsd_cases_meta)} subjects, {vsd_knees} knees\n"
    f"Ruikar (fractured): {len(valid_cases)} cases, {frac_knees} knees\n"
    f"Total volumes:      {vsd_knees + frac_knees}\n"
    f"\nVSD XY spacing:  {min(vsd_sp_xy):.3f}-{max(vsd_sp_xy):.3f}mm\n"
    f"Ruikar XY spacing: {min(frac_sp_xy):.3f}-{max(frac_sp_xy):.3f}mm\n"
    f"VSD Z spacing:   {min(vsd_sp_z):.3f}-{max(vsd_sp_z):.3f}mm\n"
    f"Ruikar Z spacing:  {min(frac_sp_z):.3f}-{max(frac_sp_z):.3f}mm\n"
    f"\nStandardization: 0.5mm isotropic, RAS"
)
axes[1, 2].text(0.05, 0.5, summary, transform=axes[1, 2].transAxes,
                fontsize=10, verticalalignment="center", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.suptitle("Cross-Dataset Comparison: Fractured vs VSD Healthy", fontsize=14)
plt.tight_layout()
fig_path = FIG_DIR / "cross_dataset_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


## Exploration Findings

### Fractured Dataset Summary
- **16 total cases** found (4 PartLeft + 12 PartRight)
- **2 excluded**: Case4 (PartLeft) and Case10 (PartRight) have only 1 Z-slice (scout/localizer)
- **14 valid cases** for downstream processing
- Variable slice thickness: 0.7mm (thin-slice) or 3.0mm (thick-slice)
- XY spacing range: ~0.37-0.55mm

### Alignment with VSD Healthy Dataset
- Both datasets use the same confirmed preprocessing parameters:
  - Resampling: 0.5mm isotropic
  - Bone window: [-450, 1050] HU
  - Orientation: RAS
  - Target volume: config-driven (128^3 local / 512^3 HPC)
- VSD scans are full lower-limb (need knee cropping + leg separation)
- Fractured scans are already knee-region (no cropping needed)

### Total Dataset
- 10 healthy knee volumes (5 VSD subjects x 2 legs)
- 14 fractured knee volumes
- **24 total volumes** (small dataset, augmentation planned)

### Next Steps
1. Run unified preprocessing notebook on both datasets
2. Generate DRRs with DiffDRR (healthy vs fractured comparison)
3. Plan augmentation strategy given the small dataset size